# Многопризнаковая регрессия на данных Diabetes

В этом учебном наборе 442 наблюдения и 10 признаков пациентов. Целевое значение — количественная оценка прогрессирования диабета через год после исходного обследования, а не диагноз «диабет / нет диабета». [Описание набора в документации scikit-learn](https://scikit-learn.org/stable/datasets/real_world.html).

В предыдущем примере `y = 2x + 1` сеть училась находить простую зависимость по одному входу. Здесь по десяти признакам нужно предсказать одно числовое значение, а зависимость не задана заранее одной формулой. Это многопризнаковая регрессия на данных пациентов.

После обучения оценим сеть на отложенных примерах и сравним её с простым прогнозом: средним целевым значением обучающей выборки.


In [1778]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from open_neuron.activations import linear, relu, gelu, prelu, elu
from open_neuron.layers import Dropout, Layer
from open_neuron.models import NeuralNetwork
from open_neuron.optimizers import Adam

np.set_printoptions(precision=3, suppress=True)
diabetes = load_diabetes()
X = diabetes.data
y = diabetes.target.reshape(-1, 1)

print("Признаки:", len(diabetes.feature_names))
print("Пример имён признаков:", diabetes.feature_names[:3])
print("Размеры X и y:", X.shape, y.shape)
print("Диапазон целевых значений:", (y.min(), y.max()))

Признаки: 10
Пример имён признаков: ['age', 'sex', 'bmi']
Размеры X и y: (442, 10) (442, 1)
Диапазон целевых значений: (np.float64(25.0), np.float64(346.0))


## Разделяем и масштабируем данные

Сначала оставляем 20% примеров для итоговой проверки, а остальные используем для обучения. `random_state=42` фиксирует разбиение: при повторном запуске те же примеры попадут в train и test.

`StandardScaler` для каждого признака вычитает среднее и делит на стандартное отклонение. Он находит эти величины только по `X_train`; к тестовым данным применяет те же найденные параметры. Поэтому средние обучающих признаков после преобразования близки к нулю, но сами значения признаков не становятся нулевыми.

Целевые значения `y` не масштабируем: сеть предсказывает их в исходной шкале, и тестовые MAE/MSE тоже остаются в этой шкале.


In [1779]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    # random_state=42,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Обучающих примеров:", len(X_train))
print("Тестовых примеров:  ", len(X_test))
print("Средние признаков train после масштабирования:",
      X_train_scaled.mean(axis=0))

Обучающих примеров: 353
Тестовых примеров:   89
Средние признаков train после масштабирования: [-0.  0. -0. -0.  0. -0. -0.  0.  0. -0.]


## Создаём и обучаем сеть

Текущая конфигурация сети:

1. На вход поступают 10 признаков.
2. `Layer(10, activation=relu)` — полносвязный скрытый слой из 10 нейронов. Каждый нейрон вычисляет взвешенную сумму входов со смещением, затем ReLU заменяет отрицательный результат нулём.
3. `Dropout(0.1)` во время обучения случайно обнуляет 10% выходов предыдущего слоя. При `predict` Dropout выключен.
4. `Layer(1, activation=linear)` выдаёт одно вещественное число без ограничения диапазона.

Сеть обучается уменьшать MSE — среднее квадратов разности между предсказаниями и ответами. Adam использует шаг обучения `0.01`. В этой ячейке сеть обучается 250 эпох: примеры перед каждой эпохой перемешиваются, размер пакета постепенно растёт от 4 до 8. `np.random.seed(7)` фиксирует случайность NumPy при инициализации весов, перемешивании и Dropout.

В `fit` не передан отдельный `validation_data`, поэтому строка `Validation Loss ... on Train Data` означает потерю, пересчитанную после эпохи на всей обучающей выборке. Это не независимая проверка. `Loss` — среднее потерь обучающих мини-пакетов, посчитанных во время обновления весов; при этих проходах Dropout активен. Поэтому две величины могут различаться.

Тестовые примеры в `fit` не передаются. Они используются ниже для оценки после обучения.


In [1780]:
# np.random.seed(7)
model = NeuralNetwork(
    [
        Layer(10, activation=relu),
        Dropout(0.2),
        Layer(1, activation=linear),
    ],
    inputs_dim=X_train_scaled.shape[1],
    loss="mse",
    optimizer=Adam(learning_rate=0.01),
)

model.fit(
    X_train_scaled,
    y_train,
    epochs=100,
    batch_size=8,
    final_batch_size=32,
    shuffle=True,
)

Training started with Overall Loss 29517.9951 on Train Data
Epoch 1/100, Loss: 28864.6134, Validation Loss: 27805.5291 on Train Data
Epoch 10/100, Loss: 5362.6021, Validation Loss: 4176.8810 on Train Data
Epoch 20/100, Loss: 4302.4792, Validation Loss: 3280.8318 on Train Data
Epoch 30/100, Loss: 4370.9197, Validation Loss: 2984.1388 on Train Data
Epoch 40/100, Loss: 3637.5217, Validation Loss: 2919.7691 on Train Data
Epoch 50/100, Loss: 3540.3626, Validation Loss: 2898.9581 on Train Data
Epoch 60/100, Loss: 3889.9901, Validation Loss: 2865.3400 on Train Data
Epoch 70/100, Loss: 4108.2536, Validation Loss: 2848.8695 on Train Data
Epoch 80/100, Loss: 3863.0702, Validation Loss: 2838.9012 on Train Data
Epoch 90/100, Loss: 3408.7525, Validation Loss: 2826.7271 on Train Data
Epoch 100/100, Loss: 4079.8274, Validation Loss: 2814.2434 on Train Data
Training completed with Overall Loss 2814.2434


## Оцениваем качество на тестовой выборке

После обучения `predict` получает масштабированные признаки `X_test_scaled` и выдаёт прогноз для каждого тестового примера. Сеть при этом работает в режиме предсказания: Dropout выключен.

- **MSE** — средний квадрат ошибки. Крупные промахи сильнее увеличивают эту метрику; её единицы — квадрат единиц целевого значения.
- **MAE** — средняя абсолютная ошибка в исходных единицах целевого значения; её проще читать как средний размер промаха.

Для сравнения базовый прогноз всегда равен среднему `y_train`. Он не использует признаки, поэтому показывает, насколько сеть полезнее постоянного ответа на этом разбиении.


In [1781]:
predictions = model.predict(X_test_scaled)
baseline_predictions = np.full_like(y_test, y_train.mean())

mse = np.mean(np.square(predictions - y_test))
mae = np.mean(np.abs(predictions - y_test))
baseline_mse = np.mean(np.square(baseline_predictions - y_test))
baseline_mae = np.mean(np.abs(baseline_predictions - y_test))

print(f"Тестовая MSE сети:       {mse:.2f}")
print(f"Тестовая MAE сети:       {mae:.2f}")
print(f"Тестовая MSE среднего:   {baseline_mse:.2f}")
print(f"Тестовая MAE среднего:   {baseline_mae:.2f}")
print()
print("Первые предсказания и ответы:")
for expected, predicted in zip(y_test.ravel()[:5], predictions.ravel()[:5]):
  print(f"ответ: {expected:.1f}, предсказание: {predicted:.1f}")

Тестовая MSE сети:       2333.61
Тестовая MAE сети:       37.14
Тестовая MSE среднего:   5449.22
Тестовая MAE среднего:   59.86

Первые предсказания и ответы:
ответ: 297.0, предсказание: 209.1
ответ: 341.0, предсказание: 246.6
ответ: 174.0, предсказание: 167.4
ответ: 152.0, предсказание: 239.5
ответ: 142.0, предсказание: 135.8


## Результат этого запуска

Эти метрики получены при следующей конфигурации:

- вход: 10 признаков;
- скрытый слой: 10 нейронов с ReLU;
- `Dropout(0.1)`;
- выход: 1 нейрон с линейной активацией;
- функция потерь: MSE;
- оптимизатор: Adam с `learning_rate=0.01`;
- 250 эпох, размер пакета постепенно меняется с 4 до 8, `shuffle=True`;
- `np.random.seed(7)`, разбиение train/test — 80/20 с `random_state=42`.

На 89 тестовых примерах сеть получила **MSE 2574.19** и **MAE 39.45**. Базовый прогноз одним средним значением обучающих ответов получил **MSE 5361.53** и **MAE 64.01**. Значит, на этом разбиении сеть предсказывает точнее базового прогноза по обеим метрикам. MAE означает, что средний абсолютный промах сети составляет около 39.45 единицы целевого показателя; MSE сильнее учитывает большие промахи, потому что ошибки возводятся в квадрат.

Первые пять примеров показывают отдельные прогнозы: для ответа `219` сеть предсказала `153.3`, а для ответа `111` — `110.7`. Средняя метрика не означает, что каждый прогноз ошибается примерно одинаково.

Эти результаты позволяют сравнивать настройки на выбранной тестовой выборке. Если выбирать конфигурацию по её метрикам, вывод относится к сравнению вариантов на этих 89 примерах.


In [1782]:
model.summary()

Info:
Network type: "Regression", layers: 2, neurons: 11, optimizer: "Adam", epochs: 100, batch size: 8-32, loss function: "Mean Squared Error"
Last forward:
Layer 1, neurons: 10, inputs: [-0.941  1.092 -0.5    1.305  0.122  0.334 -1.225  1.584  0.905  0.33 ], activation: relu, output (a): [ 4.436  6.6    4.869  3.125  6.4   10.17   3.773  6.804  5.723  0.   ]
Unit 1/10, inputs: [-0.941  1.092 -0.5    1.305  0.122  0.334 -1.225  1.584  0.905  0.33 ], weights: [-0.952 -2.724  0.964  0.015  0.151  0.888 -0.935 -0.078  1.174 -0.974], bias: [4.899], z: 4.435930893373886, activation: relu, a: 4.435930893373886
Unit 2/10, inputs: [-0.941  1.092 -0.5    1.305  0.122  0.334 -1.225  1.584  0.905  0.33 ], weights: [ 0.149 -0.016 -0.738  0.477  0.249  0.781  0.172 -0.111  0.611 -0.643], bias: [5.521], z: 6.600025888431403, activation: relu, a: 6.600025888431403
...
Unit 9/10, inputs: [-0.941  1.092 -0.5    1.305  0.122  0.334 -1.225  1.584  0.905  0.33 ], weights: [-0.352 -0.172  1.94   0.959 -1.